# 01 — Data Preparation: Invoice Extraction Dataset

**Goal:** Build a training dataset for fine-tuning Mistral 7B on invoice field extraction.

**Approach:**
1. Explore the raw HuggingFace dataset to understand what we're working with
2. Extract labeled records that already have structured annotations
3. Use GPT-4o-mini to label the remaining unlabeled OCR records
4. Generate synthetic invoices for diversity
5. Merge, split, and save as instruction-tuning JSONL

## 1. Setup

In [2]:
!pip install -q transformers datasets pydantic openai rapidfuzz python-dotenv Pillow

In [3]:
import os, sys, json, ast
from collections import Counter
from dotenv import load_dotenv

sys.path.insert(0, '..')
load_dotenv()

True

---
## 2. Explore the Raw Dataset

Before writing any processing code, let's understand what the HuggingFace dataset actually contains.

In [4]:
from datasets import load_dataset

ds = load_dataset('mychen76/invoices-and-receipts_ocr_v1', split='train')
ds = ds.remove_columns(['image'])  # skip image decoding
print(f'Total records: {len(ds)}')
print(f'Columns: {ds.column_names}')

/Users/hompushparajmehta/Desktop/job/llm-fine-tuning/llm-fine-tuning/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total records: 2043
Columns: ['id', 'parsed_data', 'raw_data']


Let's look at one record to understand the structure:

In [5]:
record = ds[0]
parsed = ast.literal_eval(record['parsed_data'])
print('parsed_data keys:', list(parsed.keys()))

inv_data = ast.literal_eval(parsed['json'])
print('\nInvoice structure keys:', list(inv_data.keys()))
print('Header:', json.dumps(inv_data['header'], indent=2))
print(f'\nItems: {len(inv_data["items"])} line items')
print('Summary:', inv_data['summary'])

parsed_data keys: ['xml', 'json', 'kie']

Invoice structure keys: ['header', 'items', 'summary']
Header: {
  "invoice_no": "40378170",
  "invoice_date": "10/15/2012",
  "seller": "Patel, Thompson and Montgomery 356 Kyle Vista New James, MA 46228",
  "client": "Jackson, Odonnell and Jackson 267 John Track Suite 841 Jenniferville, PA 98601",
  "seller_tax_id": "958-74-3511",
  "client_tax_id": "998-87-7723",
  "iban": "GB77WRBQ31965128414006"
}

Items: 1 line items
Summary: {'total_net_worth': '$7,50', 'total_vat': '$0,75', 'total_gross_worth': '$8,25'}


Now let's check how many records actually have filled-in labels vs empty ones:

In [6]:
labeled = 0
unlabeled = 0

for rec in ds:
    pd = ast.literal_eval(rec['parsed_data'])
    inv = ast.literal_eval(pd['json'])
    if inv.get('header', {}).get('seller'):
        labeled += 1
    else:
        unlabeled += 1

print(f'Records with labels: {labeled}')
print(f'Records WITHOUT labels (empty header): {unlabeled}')
print(f'\nConclusion: Only {labeled}/{len(ds)} records have structured annotations.')

Records with labels: 415
Records WITHOUT labels (empty header): 1628

Conclusion: Only 415/2043 records have structured annotations.


Let's verify that ALL records have OCR text even if labels are empty:

In [7]:
has_ocr = 0
for rec in ds:
    rd = ast.literal_eval(rec['raw_data'])
    words = ast.literal_eval(rd.get('ocr_words', '[]'))
    if len(words) > 10:
        has_ocr += 1

print(f'Records with OCR text: {has_ocr}/{len(ds)}')
print('All records have OCR text — we can label the rest with an LLM.')

Records with OCR text: 2040/2043
All records have OCR text — we can label the rest with an LLM.


**Finding:** We have 2,043 invoice images with OCR text, but only ~417 have structured labels. The remaining ~1,626 have OCR text but no extracted fields. Our plan:
1. Load the 417 pre-labeled records
2. Use GPT-4o-mini to label the 1,626 unlabeled ones
3. Generate synthetic data for additional diversity

---
## 3. Load Pre-Labeled Records

These records already have structured `header`, `items`, and `summary` fields. Our loader normalizes them to our `Invoice` schema.

In [8]:
from src.data.existing_loader import load_existing_dataset

existing_pairs = load_existing_dataset(max_samples=2000)
print(f'Loaded {len(existing_pairs)} pre-labeled invoice pairs')

Loaded 409 pre-labeled invoice pairs


Preview one sample — OCR text (input) and extracted fields (label):

In [9]:
text, invoice = existing_pairs[0]
print('=== OCR Text (first 300 chars) ===')
print(text[:300])
print('\n=== Extracted Fields ===')
print(invoice.model_dump_json(indent=2))

=== OCR Text (first 300 chars) ===
Invoice no: 40378170 Date of issue: 10/15/2012 Seller: Client: Patel, Thompson and Montgomery Jackson, Odonnell and Jackson. 356 Kyle Vista 267 John Track Suite 841 New James, MA 46228 Jenniferville, PA 98601 Tax Id: 958-74-3511 Tax Id: 998-87-7723 IBAN: GB77WRBQ31965128414006 ITEMS UM No. Descripti

=== Extracted Fields ===
{
  "vendor_name": "Patel, Thompson and Montgomery",
  "invoice_number": "40378170",
  "invoice_date": "10/15/2012",
  "due_date": "",
  "total_amount": 8.25,
  "currency": "USD",
  "tax_amount": 0.75,
  "discount": null,
  "billing_address": "Jackson, Odonnell and Jackson 267 John Track Suite 841 Jenniferville, PA 98601",
  "payment_terms": null,
  "line_items": [
    {
      "description": "Leed's Wine Companion Bottle Corkscrew Opener Gift Box Set with Foil Cutter",
      "quantity": 1.0,
      "unit_price": 7.5,
      "line_total": 8.25
    }
  ]
}


---
## 4. Label Unlabeled Records with GPT-4o-mini

The ~1,626 records without labels still have real OCR text from invoice scans. This is valuable training data — real-world OCR noise, varied formatting, different vendors. We use Azure GPT-4o-mini to extract structured fields from the OCR text.

This step saves progress incrementally — safe to interrupt and resume.

In [10]:
from src.data.label_with_llm import load_unlabeled_ocr_texts, extract_fields, get_client

unlabeled_texts = load_unlabeled_ocr_texts()
print(f'Found {len(unlabeled_texts)} unlabeled OCR records to label')

Found 1623 unlabeled OCR records to label


In [11]:
import os                                                                                                                                                            
print(f"Working directory: {os.getcwd()}")                                                                                                                           
print(f"File exists: {os.path.exists('../data/llm_labeled.jsonl')}")                                                                                                 
print(f"Also check: {os.path.exists('data/llm_labeled.jsonl')}")   

Working directory: /Users/hompushparajmehta/Desktop/job/llm-fine-tuning/notebooks
File exists: True
Also check: False


In [15]:
client = get_client()
output_path = '../data/llm_labeled.jsonl'
os.makedirs('../data', exist_ok=True)

# Load already-labeled texts to skip them
already_labeled = set()
if os.path.exists(output_path):
    with open(output_path) as f:
        for line in f:
            rec = json.loads(line)
            already_labeled.add(rec['input'][:100])
    print(f'Found {len(already_labeled)} already labeled — will skip these')
else:
    print('Starting fresh')

Found 1531 already labeled — will skip these


In [16]:
import time
from src.data.schema import Invoice

labeled, failed, skipped = 0, 0, 0
with open(output_path, 'a') as f:
    for i, ocr_text in enumerate(unlabeled_texts):
        if ocr_text[:100] in already_labeled:
            skipped += 1
            continue
        invoice = extract_fields(ocr_text, client)
        if invoice:
            record = {'instruction': 'Extract all invoice fields from the following invoice text as JSON.',
                      'input': ocr_text, 'output': invoice.model_dump_json(indent=2)}
            f.write(json.dumps(record) + '\n')
            f.flush()
            labeled += 1
        else:
            failed += 1
        if (labeled + failed) % 100 == 0 and (labeled + failed) > 0:
            print(f'Progress: {i+1}/{len(unlabeled_texts)} | New: {labeled} | Failed: {failed} | Skipped: {skipped}')
        time.sleep(0.2)

print(f'\nDone: {labeled} newly labeled, {failed} failed, {skipped} skipped (already done)')


Done: 9 newly labeled, 53 failed, 1561 skipped (already done)


Load all LLM-labeled pairs (including any from previous runs):

In [17]:
llm_labeled = []
with open(output_path) as f:
    for line in f:
        rec = json.loads(line)
        inv = Invoice.model_validate_json(rec['output'])
        llm_labeled.append((rec['input'], inv))

print(f'Total LLM-labeled: {len(llm_labeled)}')

Total LLM-labeled: 1566


---
## 5. Generate Synthetic Data

Real OCR data gives us authentic noise and formatting, but may lack diversity in currencies, date formats, and invoice styles. Synthetic data fills these gaps.

In [ ]:
from src.data.synthetic_gen import generate_dataset

synthetic_data = generate_dataset(total=1500, batch_size=5)
print(f'Generated {len(synthetic_data)} synthetic invoices')

Preview one synthetic sample:

In [ ]:
text, invoice = synthetic_data[0]
print('=== Synthetic Invoice Text ===')
print(text[:400])
print('\n=== Extracted Fields ===')
print(invoice.model_dump_json(indent=2))

In [23]:
from src.data.format import load_jsonl                                                                                                                               
from src.data.schema import Invoice                       
                                                                                                                                                                       
# Load synthetic data from backup (already generated earlier)
backup_data = load_jsonl('../data/backup/train_original.jsonl')                                                                                                      
                                                            
synthetic_data = []                                                                                                                                                  
for rec in backup_data:
    inv = Invoice.model_validate_json(rec['output'])                                                                                                                 
    synthetic_data.append((rec['input'], inv))            
                                                                                                                                                                       
print(f'Loaded {len(synthetic_data)} samples from backup')

Loaded 1134 samples from backup


---
## 6. Combine All Sources & Split

We now have three data sources. Let's merge them, deduplicate, and split into train/eval.

In [24]:
all_real = existing_pairs + llm_labeled

print(f'Pre-labeled (HF dataset):  {len(existing_pairs)}')
print(f'LLM-labeled (GPT-4o-mini): {len(llm_labeled)}')
print(f'Synthetic:                 {len(synthetic_data)}')
print(f'Total real-world:          {len(all_real)}')
print(f'Grand total:               {len(all_real) + len(synthetic_data)}')

Pre-labeled (HF dataset):  409
LLM-labeled (GPT-4o-mini): 1566
Synthetic:                 1134
Total real-world:          1975
Grand total:               3109


In [25]:
from src.data.merge import merge_and_split

train_data, eval_data = merge_and_split(all_real, synthetic_data, eval_size=500, seed=42)
print(f'Train: {len(train_data)} samples')
print(f'Eval:  {len(eval_data)} samples')

Train: 2524 samples
Eval:  500 samples


---
## 7. Format & Save

Convert to instruction-tuning format (instruction + input + output) and save as JSONL.

In [26]:
from src.data.format import format_dataset, save_jsonl

train_formatted = format_dataset(train_data)
eval_formatted = format_dataset(eval_data)

save_jsonl(train_formatted, '../data/train.jsonl')
save_jsonl(eval_formatted, '../data/eval.jsonl')

print(f'Saved train: {len(train_formatted)} samples')
print(f'Saved eval:  {len(eval_formatted)} samples')

Saved train: 2524 samples
Saved eval:  500 samples


Preview one formatted example:

In [27]:
print(json.dumps(train_formatted[0], indent=2)[:800])

{
  "instruction": "Extract all invoice fields from the following invoice text as JSON.",
  "input": "Thecooperative food Cureral England Co-ocoraivo Central Ergland Co-operative VAT508037563 OPER10R:999004 PR EO-OP WHOLEMERL LOR 0.75 PR CO-OP SOFT CHSE 1.05 PR 4.25 PR CP RASPEERRIES PUNN 2.00 PR GOAHEAD CRISPY FRUI 1.00 PR CP BBY.JERSEY ROYALS 1.00 PR NAP TOM PASSATA 0.59 PR CO-OP BAGUETTES 1.39 PR NAP TOM PASSATA 0.59 AMOUNT DUE 12.62 VF Credit Card 12.62 ACCOUNT NUMBER **** **9378 ATH003773 CHANGE 0.00 Please follou us on Titter @ycoopfood 12/05/201511:360626024 0032999004 Visit us at ww.centralengland.coop",
  "output": "{\n  \"vendor_name\": \"Thecooperative food Cureral England Co-ocoraivo Central Ergland Co-operative\",\n  \"invoice_number\": \"999004\",\n  \"invoice_date\": \"2015-


**Next:** Run `02_data_cleaning.ipynb` to analyze and clean the data before training.